In [ ]:
import sys
from pathlib import Path
# Get project root (current working directory when running from project root)
# If running from research/, use parent; otherwise use cwd()
project_root = Path.cwd() if (Path.cwd() / 'vault').exists() else Path.cwd().parent
sys.path.insert(0, str(project_root))

In [ ]:
import time
from itertools import product
from utils.enums import Ticker, TimeFrame, Direction
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from feature_extraction.feature_extractor import extract_features_for_bias_node
from feature_selection.base_models import BaseModel, QuantileBinningModel, TwoBinBinningModel
from ensemble.vault_manager import (
    initialize_vault, create_ensemble_directory,
    load_feature_base_models, list_features
)
from eda.feature_explorer import FeatureExplorer
from research.bias_node_helpers import test_bias_node, generate_node_tearsheet, get_binning_model
import utils.helpers as helpers

# RSI Bias Node Test - Lookback = 2

This notebook demonstrates the simplified workflow using the new helper functions:
1. **One-line feature extraction**: Using `extract_features_for_bias_node()` to extract features with forward returns automatically
2. **Self-contained exploration**: Using `FeatureExplorer` which handles binning model fitting internally
3. **Comprehensive analysis**: Using `generate_summary_report()` to run all relevant analysis methods at once
4. **Vault operations**: Creating and saving BaseModel to vault for production use

## Key Improvements:
- **Simplified workflow**: No manual forward returns calculation or BaseModel fitting for exploration
- **Automatic alignment**: Features and targets are automatically aligned
- **Self-contained**: FeatureExplorer handles model fitting internally
- **Comprehensive**: Single method call generates full analysis report

In [ ]:
# =============================================================================
# CONFIGURATION - Edit this cell to customize your single-node test
# =============================================================================

ENSEMBLE_TICKERS = [
    Ticker.ES,  # E-Mini S&P
    Ticker.NQ,  # E-Mini Nasdaq
    Ticker.YM,  # E-Mini Dow
    Ticker.RTY,  # E-Mini Russell 2000

    # Energy
    Ticker.CL,  # Crude Oil (WTI)
    Ticker.HO,  # ULSD (Heating Oil)

    # Metals
    Ticker.GC,  # Gold
    Ticker.HG,  # Copper
    Ticker.SI,  # Silver
    Ticker.PL,  # Platinum

    # Currencies (FX)
    Ticker.EU,  # Euro Currency
    Ticker.JY,  # Japanese Yen
    Ticker.BP,  # British Pound
    Ticker.CD,  # Canadian Dollar
    Ticker.SF,  # Swiss Franc

    # Agricultural (Food Grains)
    Ticker.C,   # Corn
    Ticker.S,   # Soybean
    Ticker.W,   # Wheat (SRW)

    # Meat
    Ticker.GF,  # Feeder Cattle

    # Fixed Income
    Ticker.TY,  # 10Yr T.Note
    Ticker.FV,  # 5Yr T.Note
    Ticker.US,  # 30Yr T.Bond
    Ticker.TU,  # 2Yr T.Note
]



START_DATE = datetime(2000, 1, 1)
END_DATE = datetime(2024, 12, 31)
PERMUTATION_REPS = 100

# Caching: set USE_CACHE=True to use pre-computed caches (~50-100x faster)
# Run once with POPULATE_CACHE=True to create caches, then set to False
USE_CACHE = True
POPULATE_CACHE = True

# Single bias node to test
bias_spec = {
    'module_name': 'ewmac', 
    'timeframes': [TimeFrame.D],
    'params': {'spanFast': [64], 'spanSlow': [256]}
}
# True for continuous features (QuantileBinning), False for rule-based (TwoBinBinning)
is_continuous = True
node_name = "ewmac"
# Strategy: 'long', 'short', or 'long-short'
STRATEGY = 'long-short'
# Target column: 'log_return', 'log_return_atr', 'log_return_ewsd'
TARGET_COL = 'log_return_atr'

reports_dir = project_root / 'research' / 'reports' / 'single_node'
reports_dir.mkdir(parents=True, exist_ok=True)

print(f"Configuration: tickers={[t.name for t in ENSEMBLE_TICKERS]}, "
      f"{START_DATE.date()} to {END_DATE.date()}, strategy={STRATEGY}, target_col={TARGET_COL}, USE_CACHE={USE_CACHE}, POPULATE_CACHE={POPULATE_CACHE}")

In [ ]:
# =============================================================================
# CACHE POPULATION (Run once with POPULATE_CACHE=True, then set to False)
# =============================================================================
if POPULATE_CACHE:
    from utils.cache_manager import CacheManager

    print("="*70)
    print("POPULATING CACHES (single node)")
    print("="*70)
    manager = CacheManager(candle_dir=str(project_root / 'data' / 'ohlc_data'))
    spec = bias_spec
    params = spec.get('params', {})
    keys = list(params.keys())
    values = [v if isinstance(v, list) else [v] for v in params.values()]
    param_combos = [dict(zip(keys, c)) for c in product(*values)] if keys else [{}]
    all_specs = [
        {'module_name': spec['module_name'], 'params': p, 'timeframes': spec.get('timeframes', [TimeFrame.D])}
        for p in param_combos
    ]
    result = manager.populate_cache(
        bias_node_specs=all_specs,
        tickers=ENSEMBLE_TICKERS,
        start_date=START_DATE,
        end_date=END_DATE,
        show_progress=True,
        overwrite_existing=False
    )
    print(f"Done: {result['success']}/{result['total']} success, {result['skipped']} skipped, {result['failed']} failed")
else:
    print("POPULATE_CACHE=False - skipping cache population")

In [ ]:
# Run full bias node test (extract, metadata, FeatureExplorer, summary report)
# Uses cache when USE_CACHE=True for ~50-100x speedup
# Optional: pass exact binning model (if None, one is chosen from is_continuous)
# binning_model = QuantileBinningModel(n_bins=10, selection_metric='sortino', strategy='long')
binning_model = TwoBinBinningModel(selection_metric='sortino', strategy=STRATEGY)



results = test_bias_node(
    bias_spec=bias_spec,
    node_name=node_name,
    is_continuous=is_continuous,
    tickers=ENSEMBLE_TICKERS,
    start_date=START_DATE,
    end_date=END_DATE,
    reports_dir=reports_dir,
    permutation_reps=PERMUTATION_REPS,
    show_plots=False,
    export=True,
    use_cache=USE_CACHE,
    binning_model=binning_model,
    strategy=STRATEGY,
    target_col=TARGET_COL
)
if results is not None:
    features_df = results['features_df']
    targets_df = results['targets_df']
    print(f"\n✓ Report exported to: {reports_dir / node_name.lower().replace(' ', '_')}")


In [ ]:
# Optional: generate QuantStats HTML tearsheet (all tickers, vs equal-weight Buy & Hold)
if results is not None:
    generate_node_tearsheet(
        node_name=node_name,
        is_continuous=is_continuous,
        start_date=START_DATE,
        end_date=END_DATE,
        reports_dir=reports_dir,
        perm_df=results.get('permutation_test'),
        features_df=results.get('features_df'),
        targets_df=results.get('targets_df'),
        strategy=STRATEGY,
        target_col=TARGET_COL,
        tickers=ENSEMBLE_TICKERS
    )


In [ ]:

# Step 7: Create BaseModel for vault operations (if needed)
# BaseModel is only needed when saving to vault
print("\n" + "="*70)
print("Creating BaseModel for vault operations...")
print("="*70)

# binning_model_vault = QuantileBinningModel(n_bins=10, selection_metric='sortino', strategy='long')
binning_model_vault = TwoBinBinningModel( selection_metric='sortino', strategy='short')

base_model = BaseModel(
    feature_config={'bias_node_spec': bias_spec},
    binning_model=binning_model_vault,
    tickers=ENSEMBLE_TICKERS
)

print("BaseModel created for vault operations!")
print(f"  Ticker: {base_model.ticker.name}")
print(f"  Bias node spec: {base_model.bias_node_spec}")
print(f"  Number of bias nodes: {len(base_model.bias_nodes)}")

# Note: BaseModel doesn't need to be fitted before saving to vault
# It can be fitted later before deployment

print("\nCreating ensemble directory...")
ensemble_dir = create_ensemble_directory(
    timeframe=TimeFrame.D,
    ensemble_name='momo_short',
    direction=Direction.SHORT,
    tickers=ENSEMBLE_TICKERS
)

print(f"✓ Created ensemble directory: {ensemble_dir}")
print(f"  Ensemble tickers: {[t.name for t in ENSEMBLE_TICKERS]}")
print(f"  Default ensemble directory is now set automatically")

In [ ]:
# Step 8: Save base model to vault
print("\nSaving base model to vault...")
# Pass all ensemble tickers when saving
# ensemble_dir is optional - uses default set by create_ensemble_directory()
model_id = base_model.save_to_vault(tickers=ENSEMBLE_TICKERS)
print(f"✓ Saved model with ID: {model_id}")
print(f"  Feature column: {base_model.feature_column}")
print(f"  Model ID: {model_id}")
print(f"  Saved with tickers: {[t.name for t in ENSEMBLE_TICKERS]}")

In [ ]:
# Step 9: Verify vault save/load
print("\nLoading models from vault...")
# ensemble_dir is optional - uses default set by create_ensemble_directory()
# Path resolution automatically handles running from research/ directory
loaded_models = load_feature_base_models(feature_column=base_model.feature_column)
model_id = 'quantile_binning_2'

print(f"✓ Loaded {len(loaded_models)} model(s)")
print(f"  Model keys (ticker, model_id): {list(loaded_models.keys())}")

# Verify loaded model (keys are now (ticker, model_id) tuples)
key = (Ticker.ES, model_id)
if key in loaded_models:
    loaded_model = loaded_models[key]
    print(f"\n✓ Model '{model_id}' for {Ticker.ES.name} loaded successfully")
    print(f"  Binning model fitted: {loaded_model.binning_model.is_fitted_}")
    print(f"  Best long bin: {loaded_model.binning_model.best_long_bin_}")
    print(f"  Ticker: {loaded_model.ticker.name}")
else:
    print(f"\n✗ Model '{model_id}' not found. Available keys: {list(loaded_models.keys())}")

# List all features in ensemble
print("\n" + "="*70)
print("Features in ensemble:")
print("="*70)
# ensemble_dir is optional - uses default set by create_ensemble_directory()
features_list = list_features()
print(features_list)